### Summarizing Mani Mama Lecture

* The youtube transcripts were not really good and there were a lot of mistakes. That is why we had to download the video and use openai-whisper library to get it transcribed. Use the transcribe.py to take the MP4 files and output the transcript into a text file.

In [1]:
import chromadb
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sqlalchemy import text


DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

def load_and_store_youtube_video(transcript_file: str):    
    # 1. Split the transcript into smaller chunks
    # Initialize the splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""]
    )
    # Load the transcript from the file
    with open(transcript_file, "r", encoding="utf-8") as f:
        transcript = f.read()
         # Use split_text to return a list of strings
        chunks = text_splitter.split_text(transcript)

        print(f"Number of chunks: {len(chunks)}")
        split_counter = 0

        # 2. Add documents to the Chroma vector store
        for chunk in chunks:
            collection.add(documents=[chunk], ids=[f"{transcript_file}_{split_counter}"])
            split_counter += 1



C:\Users\SowmyaVenky\AppData\Local\Temp\ipykernel_9748\392900146.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


In [2]:
# Ingest all the documents for dhyana slokas and chapter 1 
video_urls = [
    "videos/001.txt",
    "videos/002.txt",
    "videos/003.txt",
    "videos/004.txt",
    "videos/005.txt"
]

for url in video_urls:
    load_and_store_youtube_video(url)

Number of chunks: 113
Number of chunks: 94
Number of chunks: 100
Number of chunks: 100
Number of chunks: 126


In [6]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)

def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 5})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(result["answer"])
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print('----------------------')
        print(doc)
        print('----------------------')

In [9]:
query = "What are indriyas?" 
query_after_getting_matched_documents(query)

Indriyas refer to the senses or sense organs in Sanskrit texts, particularly in the context of Ayurveda and philosophical literature like the Bhagavad Gita. They typically include:

1. **Jnana Indriyas (Perceptual Senses)**: These are the senses through which we perceive external objects:
   - Eyes (for seeing)
   - Ears (for hearing)
   - Nose (for smelling)
   - Tongue (for tasting)
   - Skin (for touch)

2. **Indriya**: Literally meaning "instrument," it also encompasses internal faculties or organs that aid in the functions related to perception, action, and control:
   - Speech
   - Willpower/Will (to act upon intentions)
   - Mind

In a broader philosophical sense, especially within Hindu texts like the Bhagavad Gita, Indriyas are often associated with the mind as one of the vital senses, emphasizing that all experiences—whether perceived through external or internal senses—are governed by consciousness and control. Thus, Indriyas collectively represent both the sensory apparatus

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

template = """
Write a concise summary of the following 

Give the summary in points
{context}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = create_stuff_documents_chain(llm, prompt)
ans = chain.invoke({'context':docs})
print(ans)